# Простые модели коллаборативной фильтрации

**Коллаборативная фильтрация** — это метод рекомендаций, который предсказывает интересы пользователя по истории взаимодействий многих пользователей с объектами.
Три основных термина:
- User — пользователь, которому подбираем рекомендации.
- Item — объект рекомендации: фильм, товар, трек.
- Interactions — взаимодействия пользователя с объектами: оценки, просмотры, клики, покупки.

<img width='700px' src='https://campusai.github.io/_ds/recommender_systems/user-item-matrix.png'>

Как это работает: если ты и другой пользователь одинаково оценили несколько фильмов, его высокая оценка нового фильма — основание рекомендовать этот фильм тебе. Это подход user-based. В подходе item-based ищут объекты, которые нравятся одним и тем же пользователям.

## Метрики

Обозначения: $R_u^K$ — первые $K$ рекомендаций пользователю $u$, $T_u$ — релевантные items из тестовых interactions. Обычно метрику считают для каждого пользователя, затем усредняют.

### Precision@K ↑
Какая доля рекомендаций релевантна:
$$
Precision@K(u)=\frac{|R_u^K\cap T_u|}{K}
$$
Из 10 рекомендаций 3 оказались релевантными → $0{,}3$. Порядок внутри списка не учитывается.

### Recall@K ↑
Какую долю релевантных объектов удалось включить в первые $K$ рекомендаций:

$$
Recall@K(u)=\frac{|R_u^K\cap T_u|}{|T_u|}
$$

Если пользователь взаимодействовал с 5 объектами в тесте, а модель включила 2 из них в топ-10, Recall@10 равен $2/5=0{,}4$.

В отличие от Precision, знаменатель — число релевантных объектов пользователя, а не длина списка рекомендаций. Порядок внутри топ-$K$ не учитывается.

### NDCG@K ↑
Насколько хорошо расположены релевантные items. Для бинарной релевантности:
$$
DCG@K=\sum_{r=1}^{K}\frac{rel_r}{\log_2(r+1)},
\qquad
NDCG@K=\frac{DCG@K}{IDCG@K}
$$
$rel_r=1$, если item на позиции $r$ релевантен, иначе $0$. $IDCG$ — DCG идеального порядка. Диапазон — от 0 до 1: попадание на первом месте ценнее попадания на десятом.

### IntraListDiversity ↑
Насколько разнообразны items внутри одного списка. Для полного списка:
$$
ILD@K(u)=\frac{2}{K(K-1)}
\sum_{a<b}d(i_a,i_b)
$$
$d$ — расстояние между признаками объектов, например жанрами. Десять похожих боевиков дадут меньше разнообразия, чем смесь жанров. Масштаб зависит от выбранного расстояния.

### Serendipity ↑
Полезная неожиданность рекомендаций. В RecTools:
$$
Serendipity@K(u)=\frac1K\sum_{r=1}^{K}
rel_r\max\left(\frac{q(i_r)-r}{M},0\right)
$$
$q(i_r)$ — место item в общем рейтинге популярности, где 1 — самый популярный; $M$ — размер каталога. Метрика награждает релевантные items, поднятые выше их позиции по популярности. Просто рекомендовать редкое недостаточно — оно должно оказаться релевантным.

### AvgRecPopularity
Средняя популярность рекомендованных items:
$$
ARP@K(u)=\frac1{|R_u^K|}\sum_{i\in R_u^K}pop(i)
$$
В RecTools $pop(i)$ — число прошлых interactions с item. При normalize=True результат дополнительно делят на общее число interactions. Большое значение означает сильный уклон в популярное; само по себе это не «лучше» или «хуже».

### MeanInvUserFreq (Novelty) ↑
Средняя новизна через редкость items. Для полного списка:
$$
MIUF@K(u)=\frac1K\sum_{i\in R_u^K}
\log_2\frac{N}{n_i}
$$
$N$ — число пользователей в истории, $n_i$ — сколько из них взаимодействовали с item. Чем item менее распространён, тем больше вклад. Это оценка новизны по популярности, а не проверка, знаком ли item конкретному пользователю.

### Intersection.
Насколько совпадают рекомендации двух моделей:
$$
Intersection@K(u)=
\frac{|R_u^K\cap B_u^K|}{|R_u^K|}
$$
$B_u^K$ — список другой модели или версии. $1$ означает полное совпадение состава, $0$ — отсутствие общих items. Порядок не учитывается; больше не обязательно лучше. Это не Jaccard: в знаменателе размер первого списка.

### SufficientReco ↑
Удалось ли выдать нужное количество рекомендаций. Пусть $m_u$ — число выданных рекомендаций в первых $K$ позициях:
- deep=False: $1$, если $m_u\ge K$, иначе $0$.
- deep=True: $\min(m_u/K,1)$.

Выдали 7 из 10 → соответственно $0$ или $0{,}7$.

### UnrepeatedReco ↑
Нет ли дубликатов внутри списка. Пусть $d_u$ — число уникальных items среди $m_u$ рекомендаций:
- deep=False: $1$, если дубликатов нет, иначе $0$.
- deep=True: $d_u/m_u$.

Список [A, B, A, C] → соответственно $0$ или $3/4$. Проверяется повтор внутри выдачи, а не повтор ранее просмотренного. Обе метрики качества выдачи.

In [9]:
# %pip install \
#     "rectools[visuals]==0.17.0" \
#     "plotly==5.24.1" \
#     "jupyterlab==3.6.8" \
#     "notebook==6.5.7" \
#     "ipywidgets==8.1.5" \
#     "jupyterlab_widgets==3.0.13" \
#     "jupyterlab_pygments==0.2.2"

In [10]:
import typing as tp
from tqdm.auto import tqdm
import warnings

import pandas as pd
import numpy as np
import scipy
from scipy import sparse

warnings.filterwarnings("ignore")
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')

In [35]:
import rectools
from rectools import AnyIds, Columns, InternalIds
from rectools.dataset import Dataset
from rectools.models.base import ModelBase, Scores
from rectools.model_selection import TimeRangeSplitter, cross_validate
from rectools.types import InternalIdsArray

from rectools.models import (
    ImplicitItemKNNWrapperModel,
    EASEModel,
    PopularModel,
    RandomModel,
    PopularInCategoryModel
)

from rectools.metrics import (
    Precision,
    NDCG,
    IntraListDiversity,
    Serendipity,
    AvgRecPopularity,
    MeanInvUserFreq,
    Intersection,
    SufficientReco,
    UnrepeatedReco,
    PairwiseHammingDistanceCalculator,
    calc_metrics
)

import implicit
from implicit.nearest_neighbours import CosineRecommender

In [70]:
print(pd.__version__)
print(np.__version__)
print(rectools.__version__)
print(implicit.__version__)
print(scipy.__version__)

2.3.3
2.5.4
0.17.0
0.7.3
1.18.1


# Датасет KION

https://github.com/irsafilo/KION_DATASET

Набор данных для рекомендаций фильмов Kion. Данные были собраны у пользователей видеостриминговой платформы МТС Kion в период с 13.03.2021 по 22.08.2022. Это выборка анонимных данных с добавлением случайного шума.

Публичная часть набора данных включает 5 476 251 взаимодействие 962 179 пользователей с 15 706 фильмами. Набор данных включает взаимодействия пользователей с фильмами и их характеристики (такие как временная информация, продолжительность просмотра и процент просмотренного), а также демографические данные пользователей и подробную метаинформацию о фильмах.

In [71]:
! curl -L -o data_original.zip https://github.com/irsafilo/KION_DATASET/raw/f69775be31fa5779907cf0a92ddedb70037fb5ae/data_original.zip
! unzip -o data_original.zip
! rm data_original.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 75.1M  100 75.1M    0     0  6767k      0  0:00:11  0:00:11 --:--:--  9.8M
Archive:  data_original.zip
  inflating: data_original/interactions.csv  
  inflating: __MACOSX/data_original/._interactions.csv  
  inflating: data_original/users.csv  
  inflating: __MACOSX/data_original/._users.csv  
  inflating: data_original/items.csv  
  inflating: __MACOSX/data_original/._items.csv  


In [72]:
interactions = pd.read_csv('data_original/interactions.csv', parse_dates=["last_watch_dt"])
users = pd.read_csv('data_original/users.csv')
items = pd.read_csv('data_original/items.csv')

In [73]:
print('[Interactions]:')
display(interactions.head(1))
print('[Users]:')
display(users.head(1))
print('[Items]:')
display(items.head(1))

[Interactions]:


,user_id,item_id,last_watch_dt,total_dur,watched_pct
0,176549,9506,2021-05-11,4250,72.000000


[Users]:


,user_id,age,income,sex,kids_flg
0,973171,age_25_34,income_60_90,М,1


[Items]:


,item_id,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords
0,10711,film,Поговори с ней,Hable con ella,"2,002.000000","драмы, зарубежные, детективы, мелодрамы",Испания,NaN,16.000000,NaN,Педро Альмодовар,"Адольфо Фернандес, Ана Фернандес, Дарио Гранди...",Мелодрама легендарного Педро Альмодовара «Пого...,"Поговори, ней, 2002, Испания, друзья, любовь, ..."


Переименуем колонки в универсальный формат для удобства

In [74]:
# названия колонок (контракт rectools)
Columns.Weight, Columns.Datetime

('weight', 'datetime')

In [75]:
# неявный (implicit) сигнал: weight = сколько секунд просмотрел фильм
interactions = interactions.rename(
    columns={'total_dur': Columns.Weight, 'last_watch_dt': Columns.Datetime}
)

## Interactions
Пользователь `user_id` посмотрел фильм `item_id` в течении `weight` времени и последняя дата просмотра была `datetime` (просмотры **агрегируются**). Просмотры назвываются в общем случае взаимодействиями

In [76]:
pd.concat([interactions.head(), interactions.tail()])

,user_id,item_id,datetime,weight,watched_pct
0,176549,9506,2021-05-11,4250,72.000000
1,699317,1659,2021-05-29,8317,100.000000
2,656683,7107,2021-05-09,10,0.000000
3,864613,7638,2021-07-05,14483,100.000000
4,964868,9506,2021-04-30,6725,100.000000
5476246,648596,12225,2021-08-13,76,0.000000
5476247,546862,9673,2021-04-13,2308,49.000000
5476248,697262,15297,2021-08-20,18307,63.000000
5476249,384202,16197,2021-04-19,6203,100.000000
5476250,319709,4436,2021-08-15,3921,45.000000


> **Агрегация теряет историю**:
> 
> Пользователь мог смотреть фильм 1 октября (20 минут) и 5 октября (40 минут) - в строку попадет `weight=3660` (сек), `datetime='2021-10-05'`. Дальше возникает проблема валидации: если `temporal_split=2021-10-03`, то:
> - отфильтровав строку по datetime, мы потеряем реальные 20 минут просмотра до границы;
> - включив строку целиком, мы добавим 40 минут из будущего — получится утечка данных.

In [77]:
print(f"Interactions shape: {interactions.shape}")
print(f"Unique users in interactions: {interactions['user_id'].nunique():_}")
print(f"Unique items in interactions: {interactions['item_id'].nunique():_}")
print(f"Min date in interactions: {interactions['datetime'].min()}")
print(f"Max date in interactions: {interactions['datetime'].max()}")

Interactions shape: (5476251, 5)
Unique users in interactions: 962_179
Unique items in interactions: 15_706
Min date in interactions: 2021-03-13 00:00:00
Max date in interactions: 2021-08-22 00:00:00


## Users
Признаки пользователя - соцдем по пользователю (предсказанный моделью). Используем в моделях, после экспериментов с более простыми моделями

In [78]:
pd.concat([users.head(), users.tail()])

,user_id,age,income,sex,kids_flg
0,973171,age_25_34,income_60_90,М,1
1,962099,age_18_24,income_20_40,М,0
2,1047345,age_45_54,income_40_60,Ж,0
3,721985,age_45_54,income_20_40,Ж,0
4,704055,age_35_44,income_60_90,Ж,0
840192,339025,age_65_inf,income_0_20,Ж,0
840193,983617,age_18_24,income_20_40,Ж,1
840194,251008,NaN,NaN,NaN,0
840195,590706,NaN,NaN,Ж,0
840196,166555,age_65_inf,income_20_40,Ж,0


In [79]:
print(f"Users shape: {users.shape}")
print(f"Unique users: {users['user_id'].nunique():_}")

Users shape: (840197, 5)
Unique users: 840_197


## Items

Признаки по айтему - их обычно достаточно много. Можно обогащать внешними источниками по желанию. Используем в моделях, после экспериментов с более простыми моделями

In [80]:
pd.concat([items.head(2), items.tail(2)])

,item_id,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords
0,10711,film,Поговори с ней,Hable con ella,"2,002.000000","драмы, зарубежные, детективы, мелодрамы",Испания,NaN,16.000000,NaN,Педро Альмодовар,"Адольфо Фернандес, Ана Фернандес, Дарио Гранди...",Мелодрама легендарного Педро Альмодовара «Пого...,"Поговори, ней, 2002, Испания, друзья, любовь, ..."
1,2508,film,Голые перцы,Search Party,"2,014.000000","зарубежные, приключения, комедии",США,NaN,16.000000,NaN,Скот Армстронг,"Адам Палли, Брайан Хаски, Дж.Б. Смув, Джейсон ...",Уморительная современная комедия на популярную...,"Голые, перцы, 2014, США, друзья, свадьбы, прео..."
15961,4538,series,Среди камней,Darklands,"2,019.000000","драмы, спорт, криминал",Россия,0.000000,18.000000,NaN,"Марк О’Коннор, Конор МакМахон","Дэйн Уайт О’Хара, Томас Кэйн-Бирн, Джудит Родд...",Семнадцатилетний Дэмиен мечтает вырваться за п...,"Среди, камней, 2019, Россия"
15962,3206,series,Гоша,NaN,"2,019.000000",комедии,Россия,0.000000,16.000000,NaN,Михаил Миронов,"Мкртыч Арзуманян, Виктория Рунцова","Добродушный Гоша не может выйти из дома, чтобы...","Гоша, 2019, Россия"


In [81]:
print(f"Items shape {items.shape}")
print(f"Unique item_id: {items['item_id'].nunique():_}")

Items shape (15963, 14)
Unique item_id: 15_963


## Train test split (Global Temporal Split)

Есть несколько подходов, как можно выполнять сплитование в RecSys. Однако, как правило, самый надежный и максимально повторяющий реальные условия использования модели, - Global Temporal Split (глобальный сплит по времени): `date <= split_date` - train, `date > split_date` - test

In [82]:
TEST_DAYS = 7  # неделя, чтобы учесть "сезонность" дней недели

In [83]:
max_date = interactions['datetime'].max()
cutoff = max_date - pd.Timedelta(days=TEST_DAYS)

train = interactions[interactions['datetime'] < cutoff]
test = interactions[interactions['datetime'] >= cutoff]

# оставляем только горячих пользователей в тесте
test = test[test['user_id'].isin(train['user_id'].unique())]

print(f"train: {train.shape}")
print(f"test: {test.shape}")

train: (4985269, 5)
test: (349088, 5)


# Готовим данные в правильном формате



### Создаем `rectools.Dataset`, куда отправляем наш train

Зачем так сложно? Почему нельзя просто работать с interactions (взаимодействиями)?

В случае простого `.head(N)` может и возможно, но другие модели рекомендательных систем требуют хранения множества дополнительных вещей, например:

- маппинга вида `исходный user_id / item_id` - `внутренний user_id / item_id`
- просмотренные айтемы для каждого юзера, которые желательно потом удалить из рекомендованных
- правильное хранения фичей в доступном для моделей формате
- и тд

Поэтому приходим к тому, что проще инкапсулировать все эти свойства в один класс и передавать его объект в другие методы

In [84]:
# фичи пока не используем - работем с коллаборативным сигналом
dataset = Dataset.construct(
    interactions_df=train,
    user_features_df=None,
    item_features_df=None
)

Внутри `rectools.Dataset` хранит интеракции уже во "внутренних" айдишниках юзеров и айтемов (int, начиная с нуля)

In [85]:
interactions.head(2)  # исходные id

,user_id,item_id,datetime,weight,watched_pct
0,176549,9506,2021-05-11,4250,72.000000
1,699317,1659,2021-05-29,8317,100.000000


In [86]:
dataset.interactions.df.head(2)  # внутренние id

,user_id,item_id,weight,datetime
0,0,0,"4,250.000000",2021-05-11
1,1,1,"8,317.000000",2021-05-29


А также маппинги

In [87]:
dataset.user_id_map.convert_to_external([0, 1])

array([176549, 699317])

In [88]:
dataset.item_id_map.convert_to_external([0, 1])

array([9506, 1659])

И имеет возможность вернуть "сырые" интеракции (смотрим, что все айди совпали)

In [89]:
dataset.get_raw_interactions().head(2)

,user_id,item_id,weight,datetime
0,176549,9506,"4,250.000000",2021-05-11
1,699317,1659,"8,317.000000",2021-05-29


Внутренние id - это индексы строк и столбцов в sparse матрице интеракций, которая используется многими моделями для обучения (коллаборативные KNN модели, линейные автоэнкодеры, матричные факторизации)

In [90]:
dataset.get_user_item_matrix()

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 4985269 stored elements and shape (896791, 15565)>

Интеракции юзера с внутренним `user_id=0`. Нули в матрице - отсутствие взаимодействий

In [91]:
dataset.get_user_item_matrix()[0].toarray()

array([[4250., 5147.,    0., ...,    0.,    0.,    0.]],
      shape=(1, 15565), dtype=float32)

In [92]:
# можно проверить, что все ОК
dataset.interactions.df.query('user_id == 0').sort_values(by='item_id', ascending=True).head(5)

,user_id,item_id,weight,datetime
0,0,0,"4,250.000000",2021-05-11
2120773,0,1,"5,147.000000",2021-05-16
3097739,0,12,773.000000,2021-06-21
4675583,0,30,"5,150.000000",2021-06-21
487143,0,49,7.000000,2021-06-17


## Классический бейзлайн - топ популярного
Базовая модель рекомендаций: всем пользователям предлагают items с наибольшим числом interactions в train

In [93]:
pop = PopularModel()
pop.fit(dataset)

pop_recs = pop.recommend(
    users=test[Columns.User].unique(),
    dataset=dataset,
    k=10,
    filter_viewed=False  # True - удаляет просмотренные айтемы из рекомендаций
)

In [94]:
# строк ровно в 10 раз больше чем юзеров в test
pop_recs

,user_id,item_id,score,rank
0,203219,10440,"187,877.000000",1
1,203219,15297,"178,630.000000",2
2,203219,9728,"117,779.000000",3
3,203219,13865,"113,875.000000",4
4,203219,4151,"85,117.000000",5
...,...,...,...,...
1205185,857162,3734,"68,835.000000",6
1205186,857162,2657,"66,017.000000",7
1205187,857162,4880,"52,909.000000",8
1205188,857162,142,"42,466.000000",9


In [95]:
pop_recs.merge(
    items[['item_id', 'title', 'genres']],
    on='item_id',
    how='left'
).head(10)

,user_id,item_id,score,rank,title,genres
0,203219,10440,"187,877.000000",1,Хрустальный,"триллеры, детективы"
1,203219,15297,"178,630.000000",2,Клиника счастья,"драмы, мелодрамы"
2,203219,9728,"117,779.000000",3,Гнев человеческий,"боевики, триллеры"
3,203219,13865,"113,875.000000",4,Девятаев,"драмы, военные, приключения"
4,203219,4151,"85,117.000000",5,Секреты семейной жизни,комедии
5,203219,3734,"68,835.000000",6,Прабабушка легкого поведения,комедии
6,203219,2657,"66,017.000000",7,Подслушано,"драмы, триллеры"
7,203219,4880,"52,909.000000",8,Афера,комедии
8,203219,142,"42,466.000000",9,Маша,"драмы, триллеры"
9,203219,6809,"39,320.000000",10,Дуров,документальное


> **Интересное наблюдение:**
>
> В `rectools` метрики `Precision`, `Recall`, `MAP`, `MRR`, `NDCG` по умолчанию считаются на бинарной метке: 1 - есть просмотр (`user_id`, `item_id`), 0 - нет просмотра (`user_id`, `item_id`). Значение `weight` при этом не учитывается. NDCG, как кажется, должна учитывать более градуированный сигнал, например, время просмотра. Потому что один фильм могли посмотреть 2 минуты и сразу выключить, а другой 2 часа. Но с бинарной меткой такие рекомендации будут одинаково хороши (на одинаковой позиции).

In [123]:
metrics_map = {
    'Precision@10': Precision(k=10),
    'nDCG@10': NDCG(k=10),
    'AvgRecPop@10': AvgRecPopularity(k=10, normalize=True),
}

In [124]:
metrics = calc_metrics(
    metrics_map,
    reco=pop_recs,
    interactions=test,
    prev_interactions=train  # нужно только для AvgRecPopularity, чтобы считать популярность по train
)

for metric, value in metrics.items():
    print(f'{metric:>15} = {value:.4f}')

   Precision@10 = 0.0305
        nDCG@10 = 0.0364
   AvgRecPop@10 = 0.0191


# Разнообразим модель - популярное с лимитом по категориям


https://rectools.readthedocs.io/en/latest/api/rectools.models.popular_in_category.PopularInCategoryModel.html

Добавим в полку рекомендаций юзеров разнообразия по жанрам

In [125]:
items["genre"] = items["genres"].str.split(",")
items[["genres", "genre"]].head()

,genres,genre
0,"драмы, зарубежные, детективы, мелодрамы","[драмы, зарубежные, детективы, мелодрамы]"
1,"зарубежные, приключения, комедии","[зарубежные, приключения, комедии]"
2,"криминал, зарубежные, триллеры, боевики, комедии","[криминал, зарубежные, триллеры, боевики, ..."
3,"драмы, зарубежные, мелодрамы","[драмы, зарубежные, мелодрамы]"
4,"драмы, спорт, советские, мелодрамы","[драмы, спорт, советские, мелодрамы]"


In [126]:
genre_feature = items[["item_id", "genre"]].explode("genre")
genre_feature.columns = ["id", "value"]
genre_feature["feature"] = "genre"
genre_feature.head()

,id,value,feature
0,10711,драмы,genre
0,10711,зарубежные,genre
0,10711,детективы,genre
0,10711,мелодрамы,genre
1,2508,зарубежные,genre


In [127]:
genre_feature = genre_feature[genre_feature['id'].isin(train['item_id'])]

In [128]:
# датасет уже с фичей жанра
dataset_feature = Dataset.construct(
    interactions_df=train,
    user_features_df=None,
    item_features_df=genre_feature,
    cat_item_features=['genre']
)

Мы хотим оценивать **Intra-List Diversity** - разнообразие фильмов внутри списка рекомендаций одному пользователю:

$$
ILD = \frac{1}{\binom{k}{2}}\sum_{i<j} d(i,j),
$$

где \(d(i,j)\) — насколько фильмы отличаются друг от друга.

Для этого нам нужны вектора айтемов, по которым мы будем оценивать их расстояния. Возьмём жанры фильмов.

In [129]:
# Вытащим дамми векторы жанров из датасета, чтобы считать на них разнообразие айтемов внутри полки
# Здесь только жанры, потому что это единственная фича для айтемов, которую мы передали в датасет

item_genre_dummies = dataset_feature.item_features.get_dense()
item_genre_dummies[0]

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.,
       0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0.], dtype=float32)

In [130]:
genres_dummies_df = pd.DataFrame(item_genre_dummies)
genres_dummies_df.head()

,0,1,2,3,4,5,6,7,8,9,...,148,149,150,151,152,153,154,155,156,157
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [131]:
# Для подачи в калькулятор расстояний из rectools мы должны в качестве индекса датафрейма поставить айди айтемов из датасета

index = dataset.item_id_map.get_external_sorted_by_internal()
print(index.shape)
print(genres_dummies_df.shape)
index

(15565,)
(15565, 158)


array([ 9506,  1659,  7107, ..., 13516, 13019, 10542], shape=(15565,))

In [132]:
# Подставляем индекс и проверяем, что всё ок
genres_dummies_df.set_index(index, inplace=True)
genres_dummies_df.head()

,0,1,2,3,4,5,6,7,8,9,...,148,149,150,151,152,153,154,155,156,157
9506,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1659,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7107,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7638,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6686,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [133]:
# Готовим метрику разнообразия внутри листа рекомендаций
distance_calculator = PairwiseHammingDistanceCalculator(genres_dummies_df)
ild = IntraListDiversity(k=10, distance_calculator=distance_calculator)

In [134]:
metrics_map = {
    'Precision@10': Precision(k=10),
    'nDCG@10': NDCG(k=10),
    'AvgRecPop@10': AvgRecPopularity(k=10, normalize=True),
    'ILD@10': ild
}

In [135]:
pop_cat = PopularInCategoryModel(category_feature='genre', n_categories=5)
pop_cat.fit(dataset_feature)

pop_recs_cat = pop_cat.recommend(
    test[Columns.User].unique(),
    dataset=dataset_feature,
    k=10,
    filter_viewed=True
)

pop_recs_cat.merge(
    items[['item_id', 'title', 'genres']],
    on='item_id',
    how='left'
).head(10)

,user_id,item_id,score,rank,title,genres
0,699317,15297,"178,630.000000",1,Клиника счастья,"драмы, мелодрамы"
1,699317,9728,"117,779.000000",2,Гнев человеческий,"боевики, триллеры"
2,699317,12192,"31,011.000000",3,Фемида видит,"драмы, детективы, комедии"
3,699317,7829,"17,604.000000",4,Поступь хаоса,"боевики, фантастика, фэнтези, приключения"
4,699317,849,"13,592.000000",5,Дебошир,"историческое, биография, криминал, драмы, спор..."
5,699317,13865,"113,875.000000",6,Девятаев,"драмы, военные, приключения"
6,699317,1844,"23,939.000000",7,Аферистка,"триллеры, комедии"
7,699317,7107,"16,279.000000",8,Девятаев,"драмы, военные, приключения"
8,699317,2657,"66,017.000000",9,Подслушано,"драмы, триллеры"
9,699317,16166,"12,664.000000",10,Зверополис,"приключения, мультфильм, детективы, комедии"


Сравним метрики

In [138]:
print('[POP_RECS]:')

metrics = calc_metrics(
    metrics_map,
    reco=pop_recs,
    interactions=test,
    prev_interactions=train  # нужно только для AvgRecPopularity, чтобы считать популярность по train
)
for metric, value in metrics.items():
    print(f'{metric:>15} = {value:.4f}')

print('\n[POP_RECS_CAT]:')

metrics = calc_metrics(
    metrics_map,
    reco=pop_recs_cat,
    interactions=test,
    prev_interactions=train
)
for metric, value in metrics.items():
    print(f'{metric:>15} = {value:.4f}')

[POP_RECS]:
   Precision@10 = 0.0305
        nDCG@10 = 0.0364
   AvgRecPop@10 = 0.0191
         ILD@10 = 2.8667

[POP_RECS_CAT]:
   Precision@10 = 0.0270
        nDCG@10 = 0.0358
   AvgRecPop@10 = 0.0108
         ILD@10 = 5.2102


**Компромисс:**
- Precision и nDCG упали
- Popularity bias снизился
- Diversity внутри листа рекомендаций увеличился

# Item KNN

[Item-based kNN](https://contest.yandex.ru/tracks/ml/recommender-systems/introduction-to-recommender-systems#item2item-rekomendacii) рекомендует объекты, похожие на те, с которыми пользователь уже взаимодействовал. Похожесть определяется по взаимодействиям пользователей, например через косинусное сходство:

$$
sim(i,j)=\frac{x_i^\top x_j}{\|x_i\|\|x_j\|}
$$

Здесь $x_i$ и $x_j$ — столбцы матрицы взаимодействий $R$: каждый из них описывает взаимодействия всех пользователей с одним объектом.

Из сходств строят разреженную item2item матрицу $S$: для каждого объекта оставляют только $k$ ближайших соседей (исключая его самого), остальные значения обнуляют. Оценка для нового объекта $i$:

$$
\hat r_{ui}=\sum_{j\in H_u}r_{uj}S_{ji}
$$

Здесь $H_u$ — история пользователя, $r_{uj}$ — вес взаимодействия, а $\hat r_{ui}$ — предсказанная оценка кандидата.

Оценки для всех пользователей и объектов можно записать одним матричным произведением:

$$
\hat R=RS
$$

Здесь $R\in\mathbb{R}^{|U|\times|I|}$ — матрица взаимодействий, $S\in\mathbb{R}^{|I|\times|I|}$ — матрица похожести, а $\hat R$ — матрица предсказанных оценок. Для каждого пользователя выбирают объекты с наибольшими оценками, при необходимости исключая уже просмотренные.

---

В `rectools` реализована обертка над `implicit.CosineRecommender` для того, чтобы можно было использовать его в формате `rectools.Dataset`

In [139]:
cosine = ImplicitItemKNNWrapperModel(CosineRecommender(K=5))
cosine.fit(dataset)

В ходе метода `.fit` была рассчитана разреженная I2I similarity матрица с ближайшими соседями

In [140]:
cosine.model.similarity

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 77229 stored elements and shape (15565, 15565)>

Посмотрим ближайших соседей к айтему с внутренним `item_id = 0`

In [141]:
cosine.model.similarity[0]

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5 stored elements and shape (1, 15565)>

In [142]:
_,  cosine_similar_items = cosine.model.similarity[0].nonzero()      # получить внутренние item_id только для тех, у которых ненулевой similarity с item_id=0
scores = cosine.model.similarity[0, cosine_similar_items].toarray()  # получить их similarity

print(f"Similar_items: {cosine_similar_items}")
print(f"Similarity score: {scores}")

Similar_items: [    0   758  1333  2607 13361]
Similarity score: [[1.         0.19141153 0.27848838 0.13393534 0.10449729]]


Посмотрим рекомендации к айтему с внутренним `item_id = 0` из интерфейса `rectools`. Видим, что их и всего 4 - специфика модели

In [155]:
zero_recs = cosine.recommend_to_items(
    k=10,
    target_items=dataset.item_id_map.convert_to_external([0]),  # получить внешний item_id (9506) для внутреннего item_id (0)
    dataset=dataset,
    filter_itself=True)

zero_recs.merge(
    items[['item_id', 'title', 'genres']],
    on='item_id',
    how='left'
).head(10)

# Несмотря на запрос 10 рекомендаций мы видим всего 4 айтема в ответе. Это специфика модели

,target_item_id,item_id,score,rank,title,genres
0,9506,7582,0.278488,1,Холодное сердце II,"фэнтези, мультфильм, музыкальные"
1,9506,1105,0.191412,2,Рапунцель: Запутанная история,"мультфильм, приключения, мюзиклы, мелодрамы, ф..."
2,9506,9138,0.133935,3,Гномы в деле!,"семейное, развлекательные, мультфильмы"
3,9506,14863,0.104497,4,День труда,"драмы, триллеры, мелодрамы"


In [156]:
# Для какого фильма рекомендации
items.query('item_id == 9506')[['item_id', 'title']]

,item_id,title
8760,9506,Холодное сердце


In [157]:
# Посчитаем рекомендации на тесте для расчёта метрик

cos_recs = cosine.recommend(
    k=10,
    users=test[Columns.User].unique(),
    dataset=dataset,
    filter_viewed=True
)

In [158]:
metrics = calc_metrics(
    metrics_map,
    reco=cos_recs,
    interactions=test,
    prev_interactions=train  # нужно только для AvgRecPopularity, чтобы считать популярность по train
)

for metric, value in metrics.items():
    print(f'{metric:>15} = {value:.4f}')

   Precision@10 = 0.0174
        nDCG@10 = 0.0234
   AvgRecPop@10 = 0.0062
         ILD@10 = 4.0615


Наблюдения:
- Сильно снизили Precision и nDCG
- Ощутимо снилизи popularity bias

In [159]:
# Только 72% юзеров имеют заполненные списки рекомендаций
# Это важно иметь ввиду!

from rectools.metrics import SufficientReco
SufficientReco(10).calc(cos_reco)

np.float64(0.7219802013060832)

# SLIM

**SLIM (Sparse Linear Methods)** предложили в 2011 году в работе [SLIM: Sparse Linear Methods for Top-N Recommender Systems](https://doi.org/10.1109/ICDM.2011.134). Идея — сохранить линейную схему Item-based kNN, но обучать связи между объектами вместо вычисления сходств по заданной формуле.

Вместо фиксированной матрицы похожести $S$, как в Item-based kNN, обучаем матрицу item2item весов: $W\in\mathbb{R}^{M\times M}$

Предсказания:

$$
\hat R=RW
$$

Обучаем $W$ так, чтобы предсказания приближали исходные взаимодействия (принцип автоэнкодера):

$$
R\approx RW
$$

Функция потерь:

$$
\min_W
\frac{1}{2}\|R-RW\|_F^2
+\frac{\beta}{2}\|W\|_F^2
+\lambda\|W\|_1
$$

при ограничениях:

$$
W\ge 0,\qquad W_{ii}=0
$$

Первое слагаемое — ошибка восстановления взаимодействий. L2-регуляризация с коэффициентом $\beta$ ограничивает величину весов, а L1-регуляризация с коэффициентом $\lambda$ делает матрицу разреженной. Неотрицательность весов оставляет только положительные вклады. Нулевая диагональ запрещает восстанавливать взаимодействие с объектом через само себя.

Модель по-прежнему считается одним из лучших бейзлайнов, который часто бьет более большие и мощные модели.

# EASE

**EASE (Embarrassingly Shallow Autoencoders)** предложили в 2019 году в работе [Embarrassingly Shallow Autoencoders for Sparse Data](https://arxiv.org/abs/1905.03375). Простая линейная модель без скрытых слоёв показала, что на ряде задач рекомендаций можно конкурировать с глубокими моделями.

Как и SLIM, EASE обучает матрицу item-item весов $W$ для восстановления взаимодействий:

$$
\hat R=RW
$$

Но убирает L1-регуляризацию и ограничение неотрицательности, оставляя только L2-регуляризацию и нулевую диагональ:

$$
\min_W
\|R-RW\|_F^2+\lambda\|W\|_F^2,
\qquad W_{ii}=0
$$

В отличие от SLIM, матрица весов обычно **плотная**, а веса могут быть **отрицательными**: взаимодействие с одним объектом способно снижать оценку другого. Такая задача имеет **аналитическое решение**, поэтому итерационное обучение не требуется. Однако работа с плотной матрицей размера $M\times M$ требует много памяти и вычислений при большом числе объектов.

Модель по-прежнему считается одним из лучших бейзлайнов, который часто бьет более большие и мощные модели.

In [160]:
ease = EASEModel()
ease.fit(dataset)

Посмотрим на матрицу схожести айтемов

In [161]:
ease.weight.shape

(15565, 15565)

Скоры схожести для айтема с внутренним `item_id = 0`

In [162]:
ease.weight[0]

array([ 0.0000000e+00, -1.4994448e-03, -5.1234313e-04, ...,
       -3.2787815e-07,  5.1457176e-09,  1.8669631e-08],
      shape=(15565,), dtype=float32)

Как с технической точки зрения считаются рекомендации на таких скорах?

In [163]:
ease_scores = ease.weight[0]
n_recs = 20

unsorted_recs_positions = ease_scores.argpartition(-n_recs)[-n_recs:]
unsorted_recs_scores = ease_scores[unsorted_recs_positions]
sorted_recs_ids = unsorted_recs_positions[unsorted_recs_scores.argsort()[::-1]]
recs_scores = ease_scores[sorted_recs_ids]

Есть ли пересечения с ближайшими соседями от itemKNN модели?

In [164]:
cosine_similar_items

array([    0,   758,  1333,  2607, 13361], dtype=int32)

In [166]:
sorted_recs_ids

array([1333,  758,  662, 3751,  318, 2954,  133,  850,  490,   11, 1033,
         72, 1087,   14,  201,  206,  486, 4162, 1082, 2839])

2 из 5 ближайших соседей попали в топ 20 EASE

In [168]:
sum(np.isin(cosine_similar_items, sorted_recs_ids))

np.int64(2)

Посмотрим, какие рекомендации даёт EASE для айтема с внутренним `item_id = 0`. За счёт dense матрицы модель выдаёт полный список рекомендаций

In [169]:
ease_reco = ease.recommend_to_items(
    k=10,
    target_items=dataset.item_id_map.convert_to_external([0]),
    dataset=dataset,
    filter_itself=True
)
ease_reco.merge(
    items[['item_id', 'title']],
    on='item_id',
    how='left'
)

,target_item_id,item_id,score,rank,title
0,9506,7582,0.272159,1,Холодное сердце II
1,9506,1105,0.120084,2,Рапунцель: Запутанная история
2,9506,13867,0.058547,3,Рио
3,9506,12741,0.049932,4,Шиммер и Шайн
4,9506,12225,0.044508,5,Фиксики
5,9506,5535,0.043157,6,Склифосовский
6,9506,14120,0.034049,7,Маша и Медведь
7,9506,5471,0.031624,8,Доктор Хаус
8,9506,6006,0.031222,9,Кости
9,9506,496,0.031054,10,Воскресший Эртугрул


Посчитаем метрики на тесте

In [170]:
ease_recs_full = ease.recommend(
    k=10,
    users=test[Columns.User].unique(),
    dataset=dataset,
    filter_viewed=True
)

In [173]:
metrics = calc_metrics(
    metrics_map,
    reco=ease_recs_full,
    interactions=test,
    prev_interactions=train  # нужно только для AvgRecPopularity, чтобы считать популярность по train
)

for metric, value in metrics.items():
    print(f'{metric:>15} = {value:.7f}')

   Precision@10 = 0.0000075
        nDCG@10 = 0.0000077
   AvgRecPop@10 = 0.0000005
         ILD@10 = 4.5802333


### Проблема

Очень низкое значение Precision для модели, которую многие называют SOTA бейзлайном без нейронок на голых интеракциях. И легко запускают на продуктах первым MVP. Что пошло не так?

Дело в том, что в качестве коллаборативного сигнала в EASE передали `weight` — длительность просмотра в секундах. Модель минимизирует квадратичную ошибку восстановления этой матрицы. Сырые длительности заставляют EASE сильнее подстраиваться под самые долгие просмотры и ослабляют относительное влияние регуляризации, хотя метрика одинаково ценит каждый угаданный просмотр.

Решение: либо считать любой ненулевой `weight` как факт взаимодействия (1), либо нормировать `weight` и обучаться на него (уже есть аналог - `watched_pct` в датасете). Второй вариант потенциально должен дать более хорошее качество по nDCG (если считать релевантность также через `watched_pct`).

In [174]:
train.head(2)

,user_id,item_id,datetime,weight,watched_pct
0,176549,9506,2021-05-11,4250,72.000000
1,699317,1659,2021-05-29,8317,100.000000


In [176]:
# в train есть только ненулевые взаимодействия, любой weight => 1
train[Columns.Weight] = 1
dataset = Dataset.construct(train)

In [177]:
ease = EASEModel()
ease.fit(dataset)
ease_reco_ones = ease.recommend(
    k=10,
    users=test[Columns.User].unique(),
    dataset=dataset,
    filter_viewed=True
)

In [179]:
metrics = calc_metrics(
    metrics_map,
    reco=ease_reco_ones,
    interactions=test,
    prev_interactions=train  # нужно только для AvgRecPopularity, чтобы считать популярность по train
)

for metric, value in metrics.items():
    print(f'{metric:>15} = {value:.4f}')

   Precision@10 = 0.0228
        nDCG@10 = 0.0244
   AvgRecPop@10 = 0.0032
         ILD@10 = 3.3527


# Строим удобную валидацию
Вручную делить на трейн и тест и считать метрики по одной для каждой обученной модели неудобно. Облегчаем себе жизнь

In [182]:
K = 10  # сколько рекомендация для юзера и @K для метрик

metrics_map = {
    f'Precision@{K}'   : Precision(k=K),
    f'nDCG@{K}'        : NDCG(k=K),
    f'AvgRecPop@{K}'   : AvgRecPopularity(k=K, normalize=True),
    f'Diversity@{K}'   : IntraListDiversity(k=K, distance_calculator=distance_calculator),
    f'Serendipity@{K}' : Serendipity(k=K),
    f'Novelty@{K}'     : MeanInvUserFreq(k=K),
    f'Intersection@{K}': Intersection(k=K),
}

metrics_map

{'Precision@10': Precision(k=10, debias_config=None, r_precision=False),
 'nDCG@10': NDCG(k=10, debias_config=None, log_base=2, divide_by_achievable=False),
 'AvgRecPop@10': AvgRecPopularity(k=10, normalize=True),
 'Diversity@10': IntraListDiversity(k=10, distance_calculator=<rectools.metrics.distances.PairwiseHammingDistanceCalculator object at 0x31fe8b4d0>),
 'Serendipity@10': Serendipity(k=10),
 'Novelty@10': MeanInvUserFreq(k=10),
 'Intersection@10': Intersection(k=10, ref_k=None)}

In [215]:
# Global Temporal Split
splitter = TimeRangeSplitter(
    test_size="7D",           # скользящее окно по 7 дней
    n_splits=2,               # 2 фолда (для ускорения расчётов)
    filter_cold_users=True,   # только hot юзеры в тесте
    filter_cold_items=True,   # только hot айтемы в тесте
    filter_already_seen=True  # дропаем из теста просмотры трейна
)

Соберём модели и запустим кросс-валидацию по выбранной нами схеме

In [216]:
models = {
    "pop": PopularModel(),
    "cat_pop": PopularInCategoryModel("genre", 5),
    "cosine": ImplicitItemKNNWrapperModel(CosineRecommender(K=20)),
    "ease": EASEModel(),
    "random": RandomModel(random_state=32)
}

In [217]:
res = cross_validate(
    dataset=dataset_feature,
    splitter=splitter,
    metrics=metrics_map,
    models=models,
    k=K,
    filter_viewed=True,
    ref_models = models.keys(),  # с чем сравнивать выдачу модели по Intersection (со всеми моделями)
    validate_ref_models=True     # оценивать каждую модель из ref_models и по остальным метрикам (не только Intersection)
)

In [218]:
# усреденение по фолдам
(
    pd.DataFrame(res['metrics'])
    .drop(columns=['i_split'])
    .groupby(by='model', sort=False, as_index=False)
    .mean()
)

,model,Precision@10,nDCG@10,Novelty@10,AvgRecPop@10,Diversity@10,Serendipity@10,Intersection@10_pop,Intersection@10_cat_pop,Intersection@10_cosine,Intersection@10_ease,Intersection@10_random
0,pop,0.035237,0.044743,3.711777,0.016710,2.742490,0.000002,1.000000,0.465640,0.335476,0.000000,0.000636
1,cat_pop,0.024627,0.033017,4.584275,0.010985,4.935075,0.000006,0.465640,1.000000,0.213370,0.000003,0.000661
2,cosine,0.026126,0.031855,9.371409,0.006725,3.677013,0.000117,0.335486,0.213377,1.000000,0.012086,0.000643
3,ease,0.000008,0.000008,18.725374,0.000001,4.570617,0.000001,0.000000,0.000003,0.012086,1.000000,0.000625
4,random,0.000166,0.000166,15.669594,0.000061,4.576872,0.000006,0.000636,0.000661,0.000643,0.000625,1.000000


Заметили, что мы на кросс-валидацию отправляем датасет, собранный на трейне без последней недели интеракций? Наш тест остаётся нашей отложенной выборкой. Мы не будем пользоваться им в этом ноутбуке, но вполне могли бы, если бы делали модель для реального сервиса

# Напишем свою модель - TransitionsModel (first order Markov chain)
Логичным бейзлайном будет предложить пользователю самый популярный фильм, который смотрят после фильма, который он посмотрел только что. По сути это цепь Маркова 1-го порядка.  

Мы будем считать количество переходов (transitions) от каждого item к следующему в истории пользователей в нашем трейне.
Счётчики всех переходов между айтемами запишем в sparse матрицу размерности `n_hot_items` x `n_hot_items`.

При построении рекомендаций мы будем подтягивать последний айтем из истории юзера и предлагать ему те айтемы, которые чаще всего бывали следующими после него.

In [219]:
# Наследуемся от rectools.models.ModelBase для использования модели через .fit() / .recommend() и в CV

class TransitionsModel(ModelBase):
    """
    Parameters
    ----------
    verbose : int, default 0
        Degree of verbose output. If 0, no output will be provided.
    """

    recommends_for_warm = False
    recommends_for_cold = False

    def __init__(
        self,
        verbose: int = 0,
    ):
        super().__init__(verbose=verbose)
        self.weight: sparse.csr_matrix

    def _fit(self, dataset: Dataset) -> None:
        
        interactions = dataset.interactions.df
        # final transition scores will be written to a csr matrix
        # internal item ids correspond to row and col indexes of the matrix
        # rows are items to calc transisions from
        # cols are items that transition were made to
        self.weight = sparse.csr_matrix((dataset.n_hot_items, dataset.n_hot_items))

        # filter users with only one interactions because they don't have transitions
        user_inter_count = interactions.groupby(Columns.User)[Columns.Item].count()
        actual = user_inter_count[user_inter_count > 1]
        interactions = interactions[
            interactions[Columns.User].isin(actual.index)
        ].sort_values(["user_id", "datetime"])

        grouped = interactions.groupby("user_id")
        for _, group in tqdm(grouped, disable=self.verbose == 0):
            session = group["item_id"].values
            for i in range(len(session)-1):
                self.weight[session[i], session[i+1]] += 1

    def _recommend_u2i(
        self,
        user_ids: InternalIdsArray,
        dataset: Dataset,
        k: int,
        filter_viewed: bool,
        sorted_item_ids_to_recommend: tp.Optional[InternalIdsArray],
    ) -> tp.Tuple[InternalIds, InternalIds, Scores]:

        user_interactions = dataset.interactions.df[
            dataset.interactions.df[Columns.User].isin(user_ids)
        ]
        last_interactions = (
            user_interactions
            .sort_values([Columns.User, Columns.Datetime])
            .groupby(Columns.User)
            .tail(1)
        )
        target_items = last_interactions[Columns.Item].unique()

        target_ids, reco_ids, scores = self._recommend_i2i(
            target_items, dataset, k, sorted_item_ids_to_recommend
        )
        last_item_scores = pd.DataFrame({
            Columns.TargetItem: target_ids,
            Columns.Item: reco_ids,
            Columns.Score: scores
        })
        unfiltered_reco = (
            last_interactions
            .rename(columns={Columns.Item: Columns.TargetItem})
            .merge(last_item_scores, "inner", Columns.TargetItem)
            .drop(Columns.TargetItem, axis=1)
        )
        if not filter_viewed:
            reco = unfiltered_reco
        else:
            reco = unfiltered_reco.merge(
                dataset.interactions.df,
                "outer",
                Columns.UserItem,
                indicator=True
            )
            reco = reco[reco["_merge"] == "left_only"]
        
        return (
            reco[Columns.User].values,
            reco[Columns.Item].values,
            reco[Columns.Score].values   
        )

    def _recommend_i2i(
        self,
        target_ids: InternalIdsArray,
        dataset: Dataset,
        k: int,
        sorted_item_ids_to_recommend: tp.Optional[InternalIdsArray],
    ) -> tp.Tuple[InternalIds, InternalIds, Scores]:
        
        similarity = self.weight[target_ids].toarray()
        if sorted_item_ids_to_recommend is not None:
            similarity = similarity[:, sorted_item_ids_to_recommend]

        n_reco = min(k, similarity.shape[1])
        unsorted_reco_positions = similarity.argpartition(-n_reco, axis=1)[:, -n_reco:]
        unsorted_reco_scores = np.take_along_axis(similarity, unsorted_reco_positions, axis=1)

        sorted_reco_positions = unsorted_reco_scores.argsort()[:, ::-1]

        all_reco_scores = np.take_along_axis(unsorted_reco_scores, sorted_reco_positions, axis=1)
        all_reco_ids = np.take_along_axis(unsorted_reco_positions, sorted_reco_positions, axis=1)

        all_target_ids = np.repeat(target_ids, n_reco)

        if sorted_item_ids_to_recommend is not None:
            all_reco_ids = sorted_item_ids_to_recommend[all_reco_ids]

        # filter zero scores
        scores = all_reco_scores.flatten()
        positive_mask = scores > 0
        scores = scores[positive_mask]
        reco_ids = all_reco_ids.flatten()[positive_mask]
        target_ids = all_target_ids[positive_mask]

        return target_ids, reco_ids, scores

In [ ]:
dataset = Dataset.construct(train)

In [224]:
train.user_id.unique()[:10_000]

array([ 176549,  699317,  656683, ...,  325129,  619013, 1087314],
      shape=(10000,))

In [248]:
# для ускорения обучения - семлпы юзеров

train_sample_users = train.user_id.unique()[:10_000]
users_intersection = len(set(train_sample_users) & set(test_sample_users))

train_sample = train[train.user_id.isin(train_sample_users)]
test_sample = test[
    test.user_id.isin(train_sample.user_id.unique())  # тестируем только пользователей с историей в обучающей выборке
]

dataset_sample = Dataset.construct(train_sample)

print(f'train_sample shape: {train_sample.shape[0]}')
print(f'test_sample shape: {test_sample.shape[0]}')
print(f'users intersection: {users_intersection}')

train_sample shape: 240632
test_sample shape: 12655
users intersection: 37


In [249]:
trn = TransitionsModel(verbose=1)
trn.fit(dataset_sample)

recs = trn.recommend(
    users=test_sample[Columns.User].unique(),
    dataset=dataset_sample,
    k=K,
    filter_viewed=True
)

  0%|          | 0/9300 [00:00<?, ?it/s]

In [250]:
metrics_map = {
    f'Precision@{K}'   : Precision(k=K),
    f'nDCG@{K}'        : NDCG(k=K),
    f'AvgRecPop@{K}'   : AvgRecPopularity(k=K, normalize=True),
    f'Diversity@{K}'   : IntraListDiversity(k=K, distance_calculator=distance_calculator),
}

In [251]:
metrics = calc_metrics(
    metrics_map,
    reco=recs,
    interactions=test_sample,
    prev_interactions=train_sample  # нужно только для AvgRecPopularity, чтобы считать популярность по train
)

for metric, value in metrics.items():
    print(f'{metric:>15} = {value:.4f}')

   Precision@10 = 0.0175
        nDCG@10 = 0.0214
   AvgRecPop@10 = 0.0027
   Diversity@10 = 3.7831


In [254]:
# Модель часто недозаполняет рекомендации (всего 33% тестовых юзеров получили полный список из 10 айтемов)
SufficientReco(k=K).calc(recs)

np.float64(0.33217512161223073)

# Дополним свою модель: Fallback пайплайн
Если мы считаем, что на продукте TransitionModel нам может помочь для решения бизнес задачи, мы всё равно можем её использовать, но должны дополнить недостающие рекомендации другим алгоритмом. Оформим в виде класса, чтобы легко валидировать такой пайплайн наравне с обычными моделями

In [255]:
class FallbackPipeline(ModelBase):

    recommends_for_warm = False
    recommends_for_cold = False

    def __init__(
        self,
        models: tp.List[ModelBase],
        verbose: int = 0
    ):
        super().__init__(verbose=verbose)
        self.models = models

    def _fit(self, dataset: Dataset) -> None:
        for model in self.models:
            model.fit(dataset)

    def recommend(
        self,
        users: AnyIds,
        dataset: Dataset,
        k: int,
        filter_viewed: bool,
        items_to_recommend: tp.Optional[AnyIds] = None,
        add_rank_col: bool = True,
        on_unsupported_targets: str = 'ignore',
        # assume_external_ids: bool = True,
        context = None,
    ) -> pd.DataFrame:

        recos = [
            model.recommend(
                users,
                dataset,
                k,
                filter_viewed,
                items_to_recommend,
                add_rank_col=False,
                # assume_external_ids=assume_external_ids
            ) for model in self.models
        ]
        final_reco = pd.concat(recos, ignore_index=True).drop_duplicates(subset=Columns.UserItem)
        final_reco = final_reco.groupby(Columns.User, sort=False).head(k)
        final_reco[Columns.Rank] = final_reco.groupby(Columns.User, sort=False).cumcount() + 1
        final_reco.sort_values([Columns.User, Columns.Rank], inplace=True)

        if not add_rank_col:
            del final_reco[Columns.Rank]

        return final_reco

In [257]:
fb = FallbackPipeline([
    TransitionsModel(verbose=1),
    PopularModel()
])
fb.fit(dataset_sample)

reco_trn_fb = fb.recommend(
    test_sample[Columns.User].unique(),
    dataset=dataset_sample,
    k=K,
    filter_viewed=True
)

  0%|          | 0/9300 [00:00<?, ?it/s]

In [260]:
# Проверим, что у юзеров все рекомендации не повторяются
UnrepeatedReco(k=K).calc(reco_trn_fb)

np.float64(1.0)

In [261]:
# Проверим, что теперь все списки для всех юзеров заполнены
SufficientReco(10).calc(reco_trn_fb)

np.float64(1.0)

In [262]:
metrics = calc_metrics(
    metrics_map,
    reco=reco_trn_fb,
    interactions=test_sample,
    prev_interactions=train_sample  # нужно только для AvgRecPopularity, чтобы считать популярность по train
)

for metric, value in metrics.items():
    print(f'{metric:>15} = {value:.4f}')

   Precision@10 = 0.0238
        nDCG@10 = 0.0259
   AvgRecPop@10 = 0.0036
   Diversity@10 = 3.8088


# Полный код валидации моделей

In [264]:
# binary_train = train.copy()
# binary_train[Columns.Weight] = 1
# binary_dataset = Dataset.construct(binary_train)

# current_dataset = binary_dataset

# test_models = {
#     "pop": PopularModel(),
#     "cosine_20": ImplicitItemKNNWrapperModel(CosineRecommender(20)),
#     "cosine_5_w_cosine_20": FallbackPipeline([
#         ImplicitItemKNNWrapperModel(CosineRecommender(5)),
#         ImplicitItemKNNWrapperModel(CosineRecommender(20))
#         ]),
#     "trn": TransitionsModel(),
#     "trn_w_cos_5_w_pop": FallbackPipeline([
#         TransitionsModel(),
#         ImplicitItemKNNWrapperModel(CosineRecommender(5)),
#         PopularModel()]),
#     "ease": EASEModel(5000)
# }

# k = 10
# metrics = {
#     "precision": Precision(k), # Это целевая метрика для задания, остальные можно убрать если они не интересны
#     "ndcg": NDCG(k),
#     "serendipity": Serendipity(k),
#     "pop_bias": AvgRecPopularity(k, normalize=True),
#     "novelty": MeanInvUserFreq(k), # Это целевая метрика для задания, остальные можно убрать если они не интересны
# }

# splitter = TimeRangeSplitter(  # time-based валидация
#     "7D",  # скользящее окно по 7 дней
#     n_splits = 1,  # 1 фолд (делаем для ускорения расчётов)
#     filter_cold_users=True,  # только hot юзеры в тесте
#     filter_cold_items=True,  # только hot айтемы в тесте
#     filter_already_seen=True  # дропаем из теста просмотры трейна
# )

# res = cross_validate(
#     current_dataset,
#     splitter,
#     metrics,
#     test_models,
#     k=10,
#     filter_viewed=True,
# )
# metrics_res = pd.DataFrame(res["metrics"])
# metrics_res

# # Результаты можно сохранить в csv для логирования экспериментов
# metrics_res.to_csv("seminar_final.csv", index=False)

In [265]:
metrics_res = pd.DataFrame(res["metrics"])
metrics_res

,model,i_split,Precision@10,nDCG@10,Novelty@10,AvgRecPop@10,Diversity@10,Serendipity@10,Intersection@10_pop,Intersection@10_cat_pop,Intersection@10_cosine,Intersection@10_ease,Intersection@10_random
0,pop,0,0.036062,0.046231,3.711954,0.016827,2.753049,0.000002,1.000000,0.463204,0.331398,0.000000,0.000642
1,cat_pop,0,0.023460,0.031808,4.612687,0.010981,4.891302,0.000005,0.463204,1.000000,0.207608,0.000005,0.000683
2,cosine,0,0.026877,0.032730,9.367326,0.006823,3.675060,0.000113,0.331415,0.207618,1.000000,0.012259,0.000656
3,ease,0,0.000007,0.000006,18.683408,0.000001,4.599287,0.000001,0.000000,0.000005,0.012257,1.000000,0.000640
4,random,0,0.000169,0.000171,15.687434,0.000061,4.575812,0.000006,0.000642,0.000683,0.000656,0.000640,1.000000
5,pop,1,0.034412,0.043255,3.711600,0.016593,2.731932,0.000002,1.000000,0.468076,0.339553,0.000000,0.000630
6,cat_pop,1,0.025794,0.034227,4.555862,0.010988,4.978849,0.000008,0.468076,1.000000,0.219133,0.000001,0.000639
7,cosine,1,0.025375,0.030980,9.375491,0.006626,3.678967,0.000121,0.339556,0.219135,1.000000,0.011914,0.000630
8,ease,1,0.000009,0.000010,18.767341,0.000001,4.541947,0.000001,0.000000,0.000001,0.011914,1.000000,0.000610
9,random,1,0.000163,0.000160,15.651755,0.000061,4.577933,0.000006,0.000630,0.000639,0.000630,0.000610,1.000000


In [281]:
metrics_res.to_pickle("metrics_res.pkl")

## Посмотрим значения метрик на графиках

In [282]:
metrics_res = pd.read_pickle("metrics_res.pkl")

In [283]:
from rectools.visuals import MetricsApp
app = MetricsApp.construct(metrics_res)

- Высокий Precision часто приводит к высокому Popularity bias, но ряд моделей здесь может выгодно отличаться
- Классификационные и ранжирующие метрики имеют высокую корреляцию, но не всегда график будет ровной линией
- Высокая новизна не означает, что модель хорошо угадывает. Нет обязатльноей связки Novelty -> Serendipity
- На продукте где popularity bias важно держать под контролем: финальную модель можно выбирать по парето-оптимуму ранжирования/классификации и Serendipity

# Векторный расчёт метрик

In [296]:
interactions = pd.read_csv('data_original/interactions.csv', parse_dates=["last_watch_dt"])
users = pd.read_csv('data_original/users.csv')
items = pd.read_csv('data_original/items.csv')

interactions = interactions.rename(
    columns={'total_dur': Columns.Weight, 'last_watch_dt': Columns.Datetime}
)

In [298]:
sample_users = [57607, 403227, 70720]
df = (
    interactions[interactions[Columns.User].isin(sample_users)]
    .sort_values("user_id")
    .reset_index(drop=True)
)

df = df[[Columns.User, Columns.Item]]
df

,user_id,item_id
0,57607,4151
1,57607,10440
2,57607,13865
3,70720,4880
4,70720,4881
5,70720,6327
6,403227,6353
7,403227,1736
8,403227,5336
9,403227,181


In [302]:
print('Users', df[Columns.User].unique())
sample_items = df[Columns.Item].unique()
print('Items', sample_items)

Users [ 57607  70720 403227]
Items [ 4151 10440 13865  4880  4881  6327  6353  1736  5336   181]


In [305]:
top_k = 5
recs = np.array([
    np.random.choice(sample_items, top_k, replace=False),
    np.random.choice(sample_items, top_k, replace=False),
    np.random.choice(sample_items, top_k, replace=False),
])
recs

array([[13865,  6327,  6353,  4880,  4151],
       [ 4881,  4880,  6327, 10440,  1736],
       [13865, 10440,  6353,   181,  4880]])

Преобразуем в длинный датафрейм и проставим ранг

In [307]:
df_recs = pd.DataFrame({
    Columns.User: np.repeat(sample_users, top_k),
    Columns.Item: recs.ravel()
})

df_recs[Columns.Rank] = df_recs.groupby(Columns.User).cumcount() + 1
df_recs

,user_id,item_id,rank
0,57607,13865,1
1,57607,6327,2
2,57607,6353,3
3,57607,4880,4
4,57607,4151,5
5,403227,4881,1
6,403227,4880,2
7,403227,6327,3
8,403227,10440,4
9,403227,1736,5


Ключевой момент. Именно ради него преобразовывали данные и именно это позволяет считать метрики быстрее. Мы получил датафрейм с тестовыми интеракциями и предсказанными рангом от модели.

In [310]:
df_recs = df.merge(df_recs, how='left', on=Columns.UserItem)
df_recs = df_recs.sort_values(by=[Columns.User, Columns.Rank])
df_recs

,user_id,item_id,rank
2,57607,13865,1.000000
0,57607,4151,5.000000
1,57607,10440,NaN
3,70720,4880,5.000000
4,70720,4881,NaN
5,70720,6327,NaN
7,403227,1736,5.000000
6,403227,6353,NaN
8,403227,5336,NaN
9,403227,181,NaN


Для каждой реальной пары (user_id, item_id) получили позицию в рекомендациях модели. NaN означает, что пары нет в списке рекомендаций. Если rank <= K, считаем это попаданием - True Positive для метрик @K.

### Precision@K

$$
Precision@3_u = \frac{\text{попадания в топ-3}}{3}
$$

In [311]:
df_recs[f'TP@3'] = df_recs['rank'] <= 3
df_recs

,user_id,item_id,rank,TP@3
2,57607,13865,1.000000,True
0,57607,4151,5.000000,False
1,57607,10440,NaN,False
3,70720,4880,5.000000,False
4,70720,4881,NaN,False
5,70720,6327,NaN,False
7,403227,1736,5.000000,False
6,403227,6353,NaN,False
8,403227,5336,NaN,False
9,403227,181,NaN,False


Посчитаем вручную Precision@3 (усредняем по юзерам): (1/3 + 0/3 + 0/3) / 3

In [314]:
(1/3 + 0/3 + 0/3) / 3

0.1111111111111111

Посчитаем через groupby

In [315]:
df_recs['TP@3/3'] = df_recs['TP@3'] / 3
df_recs

,user_id,item_id,rank,TP@3,TP@3/3
2,57607,13865,1.000000,True,0.333333
0,57607,4151,5.000000,False,0.000000
1,57607,10440,NaN,False,0.000000
3,70720,4880,5.000000,False,0.000000
4,70720,4881,NaN,False,0.000000
5,70720,6327,NaN,False,0.000000
7,403227,1736,5.000000,False,0.000000
6,403227,6353,NaN,False,0.000000
8,403227,5336,NaN,False,0.000000
9,403227,181,NaN,False,0.000000


In [329]:
precision_at_3 = df_recs.groupby(Columns.User)['TP@3/3'].sum().mean()
precision_at_3

np.float64(0.1111111111111111)

Используем тот факт, что мы знаем количество пользователей, а значит groupby не нужен

In [331]:
precision_at_3 = df_recs['TP@3/3'].sum() / len(sample_users)
precision_at_3

np.float64(0.1111111111111111)

### Recall@K

$$
Recall@3_u = \frac{\text{попадания в топ-3}}{\text{все реальные просмотры пользователя}}
$$

In [332]:
# сколько реально просмотров у пользователя
df_recs['actual'] = df_recs.groupby(Columns.User)[Columns.Item].transform('count')
df_recs

,user_id,item_id,rank,TP@3,TP@3/3,actual,TP@3/actual
2,57607,13865,1.000000,True,0.333333,3,0.333333
0,57607,4151,5.000000,False,0.000000,3,0.000000
1,57607,10440,NaN,False,0.000000,3,0.000000
3,70720,4880,5.000000,False,0.000000,3,0.000000
4,70720,4881,NaN,False,0.000000,3,0.000000
5,70720,6327,NaN,False,0.000000,3,0.000000
7,403227,1736,5.000000,False,0.000000,4,0.000000
6,403227,6353,NaN,False,0.000000,4,0.000000
8,403227,5336,NaN,False,0.000000,4,0.000000
9,403227,181,NaN,False,0.000000,4,0.000000


In [333]:
(1/3 + 0/3 + 0/4) / 3

0.1111111111111111

In [334]:
df_recs['TP@3/actual'] = df_recs['TP@3'] / df_recs['actual']
df_recs

,user_id,item_id,rank,TP@3,TP@3/3,actual,TP@3/actual
2,57607,13865,1.000000,True,0.333333,3,0.333333
0,57607,4151,5.000000,False,0.000000,3,0.000000
1,57607,10440,NaN,False,0.000000,3,0.000000
3,70720,4880,5.000000,False,0.000000,3,0.000000
4,70720,4881,NaN,False,0.000000,3,0.000000
5,70720,6327,NaN,False,0.000000,3,0.000000
7,403227,1736,5.000000,False,0.000000,4,0.000000
6,403227,6353,NaN,False,0.000000,4,0.000000
8,403227,5336,NaN,False,0.000000,4,0.000000
9,403227,181,NaN,False,0.000000,4,0.000000


In [336]:
recall_at_3 = df_recs.groupby(Columns.User)['TP@3/actual'].sum().mean()
recall_at_3

np.float64(0.1111111111111111)

In [337]:
recall_at_3 = df_recs['TP@3/actual'].sum() / len(sample_users)
recall_at_3

np.float64(0.1111111111111111)

## Имплементация важна! Naive vs Vector

In [338]:
df = interactions[Columns.UserItem]
df.shape

(5476251, 2)

In [340]:
target = df.values
target.shape

(5476251, 2)

In [341]:
target[target[:, 0] == 176549][:, 1]  # таргет для 1 случайного юзера

array([ 9506, 15469,  9164, 12250,  5695, 11345,  7582, 10881,  5051,
        7453,  3258,  7626, 13243, 10761,  5600,  5533, 16197, 12302,
        6626,  9352, 10605,  1343,  8972,  6649, 11919,  1132,   899,
        5087, 14684,  4387,  4756, 15096,  1659,  1641, 10954,  2358,
       13018,  1105, 13424, 10732,  4360, 14689,  8211, 12324,   349,
       11654, 14006,  2956,  8437, 12770,  2722,   149, 10688, 14217,
        8283,  7000,  3182, 12104,  5171,  5411, 15915,  5956,  3834,
       11494,  6870, 15719,  2505, 15464, 14317, 11689, 10544, 15531,
       12448,  9728,   101,  7102, 11539, 16166,  4718,  4273,  7401,
       14470])

In [342]:
recs

array([[13865,  6327,  6353,  4880,  4151],
       [ 4881,  4880,  6327, 10440,  1736],
       [13865, 10440,  6353,   181,  4880]])

In [347]:
def precision_naive(df, users, recs, k):
    target = df.values
    precision = []
    for i, user in enumerate(users):
        p = 0
        user_target = target[target[:, 0] == user][:, 1]
        for rec in recs[i]:
            if rec in user_target:
                p += 1
        precision.append(p / k)
    return sum(precision) / len(users)

In [348]:
precision_naive(df, sample_users, recs, 5)

0.26666666666666666

In [349]:
def precision_pandas(df, users, recs, k):
    df_recs = pd.DataFrame({
        Columns.User: np.repeat(users, k),
        Columns.Item: recs.ravel()
    })
    df_recs["rank"] = df_recs.groupby(Columns.User).cumcount() + 1
    df_recs = df.merge(df_recs, how='left', left_on=Columns.UserItem, right_on=Columns.UserItem)
    tp_k = f'TP@{k}'
    df_recs[tp_k] = ~df_recs["rank"].isna()
    p = df_recs[tp_k].sum() / k / len(users)
    return p

In [350]:
precision_pandas(df, sample_users, recs, 5)

np.float64(0.26666666666666666)

Посмотрим через `timeit` на больших данных

In [351]:
def generate_subsample(users_count, top_k):
    users = np.random.choice(interactions[Columns.User].unique(), users_count, replace=False)
    df = interactions[interactions[Columns.User].isin(users)].reset_index(drop=True)
    del df[Columns.Datetime], df[Columns.Weight], df['watched_pct']

    recs = np.random.choice(df[Columns.Item], size=(users_count, top_k))
    return df, users, recs

In [352]:
top_k = 10
big_df, big_users, big_recs = generate_subsample(10000, top_k)

In [353]:
%timeit precision_naive(big_df, big_users, big_recs, top_k)

1.41 s ± 1.76 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [354]:
%timeit precision_pandas(big_df, big_users, big_recs, top_k) # разница почти в 10 раз

13.9 ms ± 509 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [356]:
# выигрыш в 100 раз
1.41 * 1000 / 13.9

101.43884892086331